In [1]:
import pandas as pd
import numpy as np
import joblib
import json

df = pd.read_csv('../data/studytrack_clustered.csv')

cluster_model = joblib.load('../models/student_cluster_model.pkl')
cluster_scaler = joblib.load('../models/cluster_scaler.pkl')
habit_model = joblib.load('../models/exam_score_predictor_habits.pkl')

with open('../models/cluster_names.json') as f:
    cluster_names = json.load(f)

with open('../models/feature_columns.json') as f:
    feature_meta = json.load(f)

print("Loaded. Clusters:", cluster_names)

Loaded. Clusters: {'0': 'Efficient Achiever', '1': 'Anxious & Disengaged', '2': 'Anxious Grinder', '3': 'Distracted Underachiever', '4': 'Peak Performer'}


In [2]:
habit_cols_for_benchmark = ['study_hours_per_day', 'social_media_hours', 'netflix_hours',
                              'sleep_hours', 'screen_time', 'exercise_frequency',
                              'motivation_level', 'stress_level', 'exam_anxiety_score',
                              'time_management_score', 'attendance_percentage']

benchmarks = {}

for cluster_id in sorted(df['cluster'].unique()):
    cluster_df = df[df['cluster'] == cluster_id]
    threshold = cluster_df['exam_score'].quantile(0.80)
    top_performers = cluster_df[cluster_df['exam_score'] >= threshold]
    benchmarks[cluster_id] = top_performers[habit_cols_for_benchmark].mean()
    print(f"Cluster {cluster_id} ({cluster_names[str(cluster_id)]}): top 20% threshold = {threshold:.1f}, n={len(top_performers)}")

benchmarks_df = pd.DataFrame(benchmarks).T
benchmarks_df.index = [cluster_names[str(i)] for i in benchmarks_df.index]
benchmarks_df

Cluster 0 (Efficient Achiever): top 20% threshold = 100.0, n=4492
Cluster 1 (Anxious & Disengaged): top 20% threshold = 99.0, n=4100
Cluster 2 (Anxious Grinder): top 20% threshold = 100.0, n=4460
Cluster 3 (Distracted Underachiever): top 20% threshold = 99.0, n=3344
Cluster 4 (Peak Performer): top 20% threshold = 100.0, n=5272


,study_hours_per_day,social_media_hours,netflix_hours,sleep_hours,screen_time,exercise_frequency,motivation_level,stress_level,exam_anxiety_score,time_management_score,attendance_percentage
Efficient Achiever,3.370787,1.612934,1.620236,7.107703,7.542854,3.696572,8.644479,4.866941,6.355521,5.453540,70.304252
Anxious & Disengaged,3.686080,1.135488,1.745415,7.160854,7.501780,3.794634,3.654146,4.743463,9.809024,5.456341,69.518049
Anxious Grinder,6.184763,2.849081,2.559417,7.127175,12.646883,3.666592,3.679596,4.742489,9.783857,5.516659,70.084753
Distracted Underachiever,3.434880,3.770245,1.421531,7.196322,9.597368,3.609151,4.199462,4.851555,9.578648,5.533523,69.829665
Peak Performer,5.565444,3.108194,2.319765,7.097591,12.034579,3.647003,8.627086,4.883042,6.372914,5.541730,69.825038


In [3]:
benchmarks_df.to_json('../models/cluster_benchmarks.json', orient='index')
print("Benchmarks saved.")

Benchmarks saved.


In [6]:
def assign_cluster(student_habits: dict) -> int:
    """Takes a student's habit values, returns their nearest cluster ID."""
    cluster_features_order = ['study_hours_per_day', 'social_media_hours', 'netflix_hours',
                               'sleep_hours', 'screen_time', 'exercise_frequency',
                               'motivation_level', 'stress_level', 'exam_anxiety_score',
                               'time_management_score']
    
    input_df = pd.DataFrame([[student_habits[f] for f in cluster_features_order]], 
                              columns=cluster_features_order)
    scaled_input = cluster_scaler.transform(input_df)
    cluster_id = cluster_model.predict(scaled_input)[0]
    return cluster_id


def compute_gaps(student_habits: dict, cluster_id: int) -> dict:
    """Compares student's habits vs their cluster's top-20% benchmark. 
    Positive gap = student is behind top performers on this metric."""
    cluster_label = cluster_names[str(cluster_id)]
    benchmark = benchmarks_df.loc[cluster_label]
    
    gaps = {}
    for col in habit_cols_for_benchmark:
        student_val = student_habits.get(col, benchmark[col])
        benchmark_val = benchmark[col]
        # For stress_level and exam_anxiety_score, LOWER is better, so flip the gap direction
        if col in ['stress_level', 'exam_anxiety_score']:
            gap = student_val - benchmark_val  # positive = student has MORE stress/anxiety than top performers (bad)
        else:
            gap = benchmark_val - student_val  # positive = student is BELOW top performers (needs improvement)
        gaps[col] = round(gap, 2)
    
    return gaps


# Quick test with a sample struggling student
test_student = {
    'study_hours_per_day': 2.0,
    'social_media_hours': 4.5,
    'netflix_hours': 2.0,
    'sleep_hours': 5.5,
    'screen_time': 10.0,
    'exercise_frequency': 1,
    'motivation_level': 3,
    'stress_level': 8,
    'exam_anxiety_score': 9,
    'time_management_score': 3,
    'attendance_percentage': 60
}

cluster_id = assign_cluster(test_student)
print(f"Assigned cluster: {cluster_names[str(cluster_id)]}")

gaps = compute_gaps(test_student, cluster_id)
print("\nGaps vs top performers in this cluster:")
for k, v in gaps.items():
    print(f"  {k}: {v:+.2f}")

Assigned cluster: Distracted Underachiever

Gaps vs top performers in this cluster:
  study_hours_per_day: +1.43
  social_media_hours: -0.73
  netflix_hours: -0.58
  sleep_hours: +1.70
  screen_time: -0.40
  exercise_frequency: +2.61
  motivation_level: +1.20
  stress_level: +3.15
  exam_anxiety_score: -0.58
  time_management_score: +2.53
  attendance_percentage: +9.83


In [7]:
# Feature importance weights from our habit-only model (Day 3)
importance_weights = {
    'study_hours_per_day': 0.133,
    'stress_level': 0.079,
    'sleep_hours': 0.070,
    'attendance_percentage': 0.069,
    'motivation_level': 0.062,
    'time_management_score': 0.060,
    'screen_time': 0.052,
    'netflix_hours': 0.049,
    'social_media_hours': 0.048,
    'exam_anxiety_score': 0.040,
    'exercise_frequency': 0.036
}

# Human-readable templates per habit — written to sound like real coaching advice, not robotic output
recommendation_templates = {
    'study_hours_per_day': {
        'high_gap': "You're studying about {gap:.1f} fewer hours per day than top performers in your peer group. Even a modest increase — say, one focused 45-minute block — tends to compound over a semester.",
        'label': "Study Time"
    },
    'stress_level': {
        'high_gap': "Your stress level is running noticeably higher than students with similar habits who perform well. High stress doesn't just feel bad — it directly eats into focus and retention.",
        'label': "Stress Management"
    },
    'sleep_hours': {
        'high_gap': "You're getting about {gap:.1f} fewer hours of sleep than your top-performing peers. Sleep is when memory consolidation happens — skimping on it undercuts the studying you're already doing.",
        'label': "Sleep"
    },
    'attendance_percentage': {
        'high_gap': "Your attendance is about {gap:.0f} percentage points below top performers in your group. Consistent attendance compounds — missed context is hard to fully recover from self-study alone.",
        'label': "Attendance"
    },
    'motivation_level': {
        'high_gap': "Your motivation levels are lower than peers who are performing well with similar study patterns. This is often the real lever — habits are easier to sustain once motivation is addressed.",
        'label': "Motivation"
    },
    'time_management_score': {
        'high_gap': "Your time management score suggests there's room to structure your study sessions more effectively — not necessarily study more, but study smarter.",
        'label': "Time Management"
    },
    'screen_time': {
        'high_gap': "Your overall screen time is higher than your top-performing peers by about {gap:.1f} hours. Worth auditing where that time actually goes.",
        'label': "Screen Time"
    },
    'social_media_hours': {
        'high_gap': "Social media usage is running higher than your peers who perform well — this is often one of the easiest, fastest wins since it's a direct time trade-off with study hours.",
        'label': "Social Media"
    },
    'netflix_hours': {
        'high_gap': "Entertainment/streaming time is above what your top-performing peers report — not a problem in moderation, but worth capping if study hours are tight.",
        'label': "Entertainment Time"
    },
    'exam_anxiety_score': {
        'high_gap': "Your exam anxiety is measurably higher than peers with strong outcomes. This often responds well to practice-testing and structured prep, not just 'staying calm.'",
        'label': "Exam Anxiety"
    },
    'exercise_frequency': {
        'high_gap': "You're exercising less frequently than your top-performing peers. Physical activity has a real, measurable link to focus and stress regulation — even light, regular movement helps.",
        'label': "Exercise"
    }
}


def generate_recommendations(gaps: dict, top_n: int = 4) -> list:
    """
    Ranks gaps by (gap size × feature importance) to prioritize the highest-impact,
    most-needed recommendations — not just the biggest raw gap.
    """
    scored_gaps = []
    for habit, gap in gaps.items():
        if gap > 0:  # only recommend on things where the student is actually behind
            weight = importance_weights.get(habit, 0.03)
            priority_score = gap * weight
            scored_gaps.append((habit, gap, priority_score))
    
    scored_gaps.sort(key=lambda x: x[2], reverse=True)
    
    recommendations = []
    for habit, gap, score in scored_gaps[:top_n]:
        template = recommendation_templates[habit]
        text = template['high_gap'].format(gap=gap)
        recommendations.append({
            'area': template['label'],
            'priority_score': round(score, 3),
            'gap': gap,
            'recommendation': text
        })
    
    return recommendations


recs = generate_recommendations(gaps)
for i, r in enumerate(recs, 1):
    print(f"{i}. [{r['area']}] {r['recommendation']}\n")

1. [Attendance] Your attendance is about 10 percentage points below top performers in your group. Consistent attendance compounds — missed context is hard to fully recover from self-study alone.

2. [Stress Management] Your stress level is running noticeably higher than students with similar habits who perform well. High stress doesn't just feel bad — it directly eats into focus and retention.

3. [Study Time] You're studying about 1.4 fewer hours per day than top performers in your peer group. Even a modest increase — say, one focused 45-minute block — tends to compound over a semester.

4. [Time Management] Your time management score suggests there's room to structure your study sessions more effectively — not necessarily study more, but study smarter.

